In [13]:
from pathlib import Path
from llama_index.readers.file import PDFReader
import os

from dotenv import load_dotenv, find_dotenv
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.core.llms import ChatMessage
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core.llms import ChatMessage

from llama_index.core.ingestion import IngestionPipeline

# Response Synthesis

In this notebook, we will explore the response synthesis module, focusing on generating responses using various modes and indices.

## Setup

In [14]:
import os

In [15]:
from dotenv import load_dotenv, find_dotenv
load_dotenv('D:/.env')

False

In [16]:
os.environ["AZURE_OPENAI_API_KEY"] = "F8kUozKumg8vOqdM6i3uF3MEnHQyAWnh5si5hgocdPdXbakenhTWJQQJ99BLACfhMk5XJ3w3AAAAACOGSVUE"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://genaifoundry766488650611.openai.azure.com/"
os.environ["OPENAI_API_VERSION"] = "2024-02-01"

In [17]:
OPENAI_API_KEY = os.environ['AZURE_OPENAI_API_KEY']

## Download Data

In [18]:
!mkdir data
!wget "https://arxiv.org/pdf/1706.03762" -O 'data/transformers.pdf'

A subdirectory or file data already exists.
'wget' is not recognized as an internal or external command,
operable program or batch file.


In [19]:
from pathlib import Path
from llama_index.readers.file import PDFReader

In [20]:
loader = PDFReader()

In [21]:
documents = loader.load_data(file=Path('./data/transformers.pdf'))

In [22]:
len(documents)

15

In [23]:
embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small"
)

embedding = embed_model.get_text_embedding("The cat sat on the mat")

# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=20),
        embed_model,
        ]
    )

# run the pipeline
nodes = pipeline.run(documents=documents)

len(nodes)

2026-01-29 15:03:07,033 - INFO - HTTP Request: POST https://genaifoundry766488650611.openai.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2024-02-01 "HTTP/1.1 404 DeploymentNotFound"


NotFoundError: Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}

In [ ]:
index = VectorStoreIndex(
    nodes,
    show_progress=True,
    embed_model=embed_model,
)

In [12]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex.from_documents(documents)

ValueError: 
******
Could not load OpenAI embedding model. If you intended to use OpenAI, please check your OPENAI_API_KEY.
Original error:
No API key found for OpenAI.
Please set either the OPENAI_API_KEY environment variable or openai.api_key prior to initialization.
API keys can be found or created at https://platform.openai.com/account/api-keys

Consider using embed_model='local'.
Visit our documentation for more embedding options: https://developers.llamaindex.ai/python/framework/module_guides/models/embeddings/
******

In [11]:
# configure retriever
retriever = index.as_retriever()

# Different types of response synthesizer

## Refine

In [12]:
from llama_index.core import get_response_synthesizer

In [14]:
# configure response synthesizer
response_synthesizer = get_response_synthesizer(response_mode="refine")

## Compact

In [15]:
# configure response synthesizer
response_synthesizer = get_response_synthesizer(response_mode="compact")

## Tree Summarize

In [16]:
# configure response synthesizer
response_synthesizer = get_response_synthesizer(response_mode="tree_summarize")

## Accumulate

In [17]:
# configure response synthesizer
response_synthesizer = get_response_synthesizer(response_mode="accumulate")

## Compact Accumulate

In [18]:
# configure response synthesizer
response_synthesizer = get_response_synthesizer(response_mode="compact_accumulate")

# Next: Setting up the Query Engine